# Hex6 Colab Autopilot

This notebook keeps Colab interaction file-driven: prepare the repo, optionally submit one default request, start one background worker, then monitor request/result/progress files.

In [ ]:
REPO_MODE = "git"  # "git" or "drive"
REPO_URL = "https://github.com/Stroudmj00/hex6-bot.git"
REPO_BRANCH = "codex/colab-autopilot-temp"  # change to "main" after the autopilot branch is merged
DRIVE_REPO_PATH = "/content/drive/MyDrive/Hex-A-Toe"
WORKDIR = "/content/hex6-bot"
RESET_WORKDIR = True

PLAN = "configs/colab_autopilot.toml"
WORKER_ID = "colab-g4-autopilot-01"
MINIMUM_GPU_TIER = "T4"
STATUS_BACKEND = "none"
WORKER_ONCE = True
WORKER_LOG = "artifacts/colab_autopilot/worker.log"
WORKER_STATE = "artifacts/colab_autopilot/worker_state.json"

SUBMIT_IF_IDLE = True
REQUEST_KIND = "cycle"
REQUEST_PRIORITY = 90
REQUEST_CONFIG = "configs/colab_strongest_v2_safe.toml"
REQUEST_OUTPUT_ROOT = "artifacts/bootstrap_colab_strongest_v2_safe"
REQUEST_MINUTES = 60.0
REQUEST_NOTES = "Safe strongest-v2 Colab retry from the autopilot notebook."

MONITOR_TAIL_LINES = 80
MONITOR_WATCH_POLLS = 1
MONITOR_INTERVAL_SECONDS = 30.0


## Prepare Runtime

Run this once after connecting a GPU runtime.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

try:
    from google.colab import userdata
    IN_COLAB = True
except Exception:
    userdata = None
    IN_COLAB = False

if userdata is not None:
    try:
        token = userdata.get("HEX6_GITHUB_TOKEN")
    except Exception:
        token = None
    if token:
        os.environ["HEX6_GITHUB_TOKEN"] = token

def run(cmd, *, cwd=None, check=True):
    print("$", " ".join(str(part) for part in cmd))
    return subprocess.run(cmd, cwd=cwd, check=check, text=True)

if MINIMUM_GPU_TIER:
    run(["nvidia-smi"], check=False)

workdir = Path(WORKDIR)
if REPO_MODE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    if RESET_WORKDIR and workdir.exists():
        shutil.rmtree(workdir)
    if not workdir.exists():
        shutil.copytree(DRIVE_REPO_PATH, WORKDIR)
else:
    if RESET_WORKDIR and workdir.exists():
        shutil.rmtree(workdir)
    if not workdir.exists():
        run(["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, WORKDIR])

os.chdir(WORKDIR)
run([sys.executable, "-m", "pip", "install", "-e", WORKDIR])
run(["git", "rev-parse", "--abbrev-ref", "HEAD"], cwd=WORKDIR, check=False)
run(["git", "rev-parse", "--short", "HEAD"], cwd=WORKDIR, check=False)
print("prepared_at", datetime.now(timezone.utc).replace(microsecond=0).isoformat())


## Submit Default Request

This submits only when there is no pending or running request.

In [ ]:
import json
import subprocess
import sys

def run_json(cmd):
    print("$", " ".join(str(part) for part in cmd))
    completed = subprocess.run(cmd, cwd=WORKDIR, check=True, capture_output=True, text=True)
    if completed.stderr.strip():
        print(completed.stderr.strip())
    print(completed.stdout.strip())
    return json.loads(completed.stdout)

list_cmd = [sys.executable, "-m", "hex6.integration.run_autopilot", "--plan", PLAN, "list"]
state = run_json(list_cmd)
active = [row for row in state.get("requests", []) if row.get("status") in {"pending", "running"}]

if SUBMIT_IF_IDLE and not active:
    submit_cmd = [
        sys.executable,
        "-m",
        "hex6.integration.run_autopilot",
        "--plan",
        PLAN,
        "submit",
        "--kind",
        REQUEST_KIND,
        "--priority",
        str(REQUEST_PRIORITY),
        "--notes",
        REQUEST_NOTES,
    ]
    if REQUEST_KIND == "cycle":
        submit_cmd.extend([
            "--config",
            REQUEST_CONFIG,
            "--output-root",
            REQUEST_OUTPUT_ROOT,
            "--minutes",
            str(REQUEST_MINUTES),
        ])
    submit_result = run_json(submit_cmd)
    print("submitted_request", submit_result.get("request_id"))
else:
    print("active_requests", [row.get("request_id") for row in active])


## Start Background Worker

This returns immediately and writes worker stdout/stderr to `artifacts/colab_autopilot/worker.log`.

In [ ]:
import json
import os
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

def pid_running(pid):
    try:
        os.kill(int(pid), 0)
    except (OSError, TypeError, ValueError):
        return False
    return True

state_path = Path(WORKDIR) / WORKER_STATE
log_path = Path(WORKDIR) / WORKER_LOG
state_path.parent.mkdir(parents=True, exist_ok=True)
log_path.parent.mkdir(parents=True, exist_ok=True)

existing = json.loads(state_path.read_text(encoding="ascii")) if state_path.exists() else {}
if existing.get("pid") and pid_running(existing["pid"]):
    print("worker_already_running", existing)
else:
    cmd = [
        sys.executable,
        "-u",
        "scripts/colab_run.py",
        "autopilot-worker",
        "--repo-root",
        WORKDIR,
        "--plan",
        PLAN,
        "--worker-id",
        WORKER_ID,
        "--status-backend",
        STATUS_BACKEND,
    ]
    if MINIMUM_GPU_TIER:
        cmd.extend(["--minimum-gpu-tier", MINIMUM_GPU_TIER])
    if WORKER_ONCE:
        cmd.append("--once")

    log_handle = log_path.open("ab", buffering=0)
    process = subprocess.Popen(cmd, cwd=WORKDIR, stdout=log_handle, stderr=subprocess.STDOUT, start_new_session=True)
    log_handle.close()
    worker_state = {
        "pid": process.pid,
        "command": cmd,
        "log_path": str(log_path),
        "started_at": datetime.now(timezone.utc).replace(microsecond=0).isoformat().replace("+00:00", "Z"),
    }
    state_path.write_text(json.dumps(worker_state, indent=2), encoding="ascii")
    print("started_worker", json.dumps(worker_state, indent=2))


## Monitor

Rerun this cell to refresh queue, progress, result, and log state.

In [ ]:
import subprocess
import sys

monitor_cmd = [
    sys.executable,
    "scripts/colab_autopilot_monitor.py",
    "--repo-root",
    WORKDIR,
    "--worker-state",
    WORKER_STATE,
    "--worker-log",
    WORKER_LOG,
    "--tail-lines",
    str(MONITOR_TAIL_LINES),
]
if MONITOR_WATCH_POLLS != 1:
    monitor_cmd.extend(["--watch", "--interval", str(MONITOR_INTERVAL_SECONDS), "--max-polls", str(MONITOR_WATCH_POLLS)])
subprocess.run(monitor_cmd, cwd=WORKDIR, check=False)


## Inspect Latest Result

Use this after the worker exits to decide whether the checkpoint goes to the ladder, needs a follow-up, or is only evidence.

In [ ]:
import json
from pathlib import Path

result_dir = Path(WORKDIR) / "artifacts/colab_autopilot/results"
results = sorted(result_dir.glob("*.json"), key=lambda path: path.stat().st_mtime)
if not results:
    print("no_result_files")
else:
    latest = results[-1]
    payload = json.loads(latest.read_text(encoding="ascii"))
    request = payload.get("request", {})
    result = payload.get("result", {})
    print("latest_result", latest)
    print("request", json.dumps({
        "request_id": request.get("request_id"),
        "kind": request.get("kind"),
        "status": request.get("status"),
        "exit_code": request.get("exit_code"),
        "error": request.get("error"),
    }, indent=2))
    print("checkpoint", result.get("checkpoint_path") or result.get("best_checkpoint") or result.get("latest_checkpoint") or "")
    print("suggested_ladder_submission", json.dumps(result.get("suggested_ladder_submission", {}), indent=2))
    if request.get("status") != "completed":
        print("decision_hint: archive failed evidence or submit a smaller follow-up request")
    elif result.get("suggested_ladder_submission"):
        print("decision_hint: review the checkpoint metrics, then create a ladder request if the result is plausible")
    else:
        print("decision_hint: archive as evidence unless the summary points to a specific follow-up")
